# Import Required Libraries
Import necessary libraries such as pandas, scikit-learn, and NLTK for data handling, machine learning, and text processing.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import matplotlib.pyplot as plt
import seaborn as sns

# Download NLTK data if needed
nltk.download('stopwords')
nltk.download('wordnet')

print("Libraries imported successfully!")

# Load and Preprocess Incident Data
Load incident ticket data from a CSV or database, clean text fields, handle missing values, and prepare the dataset for analysis.

In [ ]:
# Load data
df = pd.read_csv('../data/incidents.csv')
print(f"Loaded {len(df)} incident tickets")
print(df.head())

# Basic preprocessing
df['description'] = df['description'].fillna('')
df['clean_description'] = df['description'].str.lower().str.replace(r'[^a-zA-Z\s]', '', regex=True)

# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words and len(token) > 2]
    return ' '.join(tokens)

df['processed_description'] = df['clean_description'].apply(preprocess_text)
print("Data preprocessing completed")
print(df[['description', 'processed_description']].head())

# Define Taxonomy Hierarchy
Define the multi-level taxonomy structure (e.g., levels like Category, Subcategory, Root Cause) and map it to the data.

In [ ]:
import json

# Load taxonomy
with open('../data/taxonomy.json', 'r') as f:
    taxonomy = json.load(f)

print("Taxonomy structure:")
print(json.dumps(taxonomy, indent=2))

# Check data columns
print("Data columns:", df.columns.tolist())

# Ensure taxonomy levels match data
taxonomy_levels = taxonomy['levels']
print(f"Taxonomy levels: {taxonomy_levels}")

# Visualize class distribution
for level in taxonomy_levels:
    level_col = level.lower()
    if level_col in df.columns:
        plt.figure(figsize=(10, 6))
        df[level_col].value_counts().plot(kind='bar')
        plt.title(f'Distribution of {level}')
        plt.xticks(rotation=45)
        plt.show()

# Extract Features from Tickets
Extract features such as TF-IDF vectors from ticket descriptions, keywords, or other attributes for model input.

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X = tfidf.fit_transform(df['processed_description'])

print(f"TF-IDF matrix shape: {X.shape}")
print(f"Number of features: {len(tfidf.get_feature_names_out())}")

# Show top features
feature_names = tfidf.get_feature_names_out()
tfidf_scores = X.sum(axis=0).A1
top_features = pd.DataFrame({'feature': feature_names, 'score': tfidf_scores})
top_features = top_features.sort_values('score', ascending=False).head(20)
print("Top TF-IDF features:")
print(top_features)

# Train Hierarchical Classification Model
Train a hierarchical classifier (e.g., using scikit-learn's MultiOutputClassifier or a custom pipeline) on the preprocessed data.

In [ ]:
from sklearn.multioutput import MultiOutputClassifier

# Prepare labels for each level
y_levels = []
for level in taxonomy_levels:
    level_col = level.lower()
    if level_col in df.columns:
        y_levels.append(df[level_col].fillna('Unknown'))
    else:
        print(f"Warning: {level_col} not found in data")

y = np.column_stack(y_levels)
print(f"Labels shape: {y.shape}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train multi-output classifier
clf = MultiOutputClassifier(LogisticRegression(random_state=42, max_iter=1000))
clf.fit(X_train, y_train)

print("Model trained successfully!")

# Predict on test set
y_pred = clf.predict(X_test)
print(f"Predictions shape: {y_pred.shape}")

# Implement Triage Function
Create a function to automatically classify new incident tickets into the taxonomy levels based on the trained model.

In [ ]:
def triage_incident(description, model, vectorizer, taxonomy_levels):
    """Triage a new incident ticket into taxonomy levels."""
    # Preprocess text
    clean_text = description.lower().replace(r'[^a-zA-Z\s]', '')
    processed_text = preprocess_text(clean_text)
    
    # Vectorize
    X_new = vectorizer.transform([processed_text])
    
    # Predict
    pred_levels = model.predict(X_new)[0]
    
    # Create result dictionary
    result = {}
    for i, level in enumerate(taxonomy_levels):
        result[level.lower()] = pred_levels[i]
    
    return result

# Test the triage function
test_description = "Database connection failed during peak hours"
result = triage_incident(test_description, clf, tfidf, taxonomy_levels)
print(f"Triage result for: '{test_description}'")
print(result)

# Evaluate Model Performance
Evaluate the model using metrics like accuracy, precision, and recall for each taxonomy level, and visualize results.

In [ ]:
# Evaluate each level
for i, level in enumerate(taxonomy_levels):
    level_col = level.lower()
    print(f"\nEvaluation for {level}:")
    
    # Get true and predicted for this level
    y_true_level = y_test[:, i]
    y_pred_level = y_pred[:, i]
    
    # Calculate metrics
    accuracy = accuracy_score(y_true_level, y_pred_level)
    print(f"Accuracy: {accuracy:.4f}")
    
    # Classification report
    print(classification_report(y_true_level, y_pred_level, zero_division=0))

# Overall hierarchical accuracy (exact match)
exact_matches = np.all(y_test == y_pred, axis=1)
hierarchical_accuracy = np.mean(exact_matches)
print(f"\nHierarchical Accuracy (exact match): {hierarchical_accuracy:.4f}")

# Visualize confusion matrix for first level
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test[:, 0], y_pred[:, 0])
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix for {taxonomy_levels[0]}')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()